# LifeGuard AI — Module 2: Fair Price (EDA, Leakage-Safe Regression & Error Analysis)

This notebook audits the CarDekho Used Car & Used Bike datasets, verifies leakage-free preprocessing (`ColumnTransformer` + `Pipeline`), compares Ridge baseline, Random Forest, XGBoost, and LightGBM on `MAE`, `RMSE`, and `R²`, and inspects error brackets on the untouched 20% test split.

In [ ]:
import json
from pathlib import Path
import pandas as pd

REPO_ROOT = Path("../../..").resolve()
META_PATH = REPO_ROOT / "ml" / "price" / "artifacts" / "metadata.json"

entries = json.loads(META_PATH.read_text(encoding="utf-8"))
print(f"Loaded {len(entries)} Fair Price model reports.")

## 1. Dataset Audit & Leakage Prevention

- **Exact Duplicates Removed First**: 763 duplicate car rows (`4,340 -> 3,577`) and 6 duplicate bike rows (`1,061 -> 1,055`) are dropped *before* creating the 80/20 stratified-by-price-quantile split so no identical vehicle appears in both train and test.
- **Missing Values Handled Inside CV Folds**: `bike_details.csv` has 433 missing `ex_showroom_price` entries (~41%). `SimpleImputer(strategy="median", add_indicator=True)` learns medians strictly inside each training fold.
- **Domain Feature Extraction**: `CarFeatureExtractor` and `BikeFeatureExtractor` derive normalized `brand`, `model_word`, `vehicle_age`, `km_per_year`, and bike `engine_cc_hint` inside the saved `Pipeline`.

In [ ]:
for entry in entries:
    print(f"=== {entry['sub_model'].upper()} TRACK ===")
    print("Dataset:", entry["dataset"])
    print("Splits :", entry["splits"])
    display(pd.DataFrame(entry["comparison_table"]))

## 2. Untouched Test Metrics, Tier Error Analysis & Limitations

In [ ]:
for entry in entries:
    print(f"=== {entry['sub_model'].upper()} TEST RESULTS ({entry['model_name']}) ===")
    print("Metrics:", entry["metrics"])
    print("Tier MAE breakdown:")
    display(pd.DataFrame(entry["error_analysis"]["tier_mae"]))
    print("Worst residuals sample:")
    display(pd.DataFrame(entry["error_analysis"]["worst_residuals_sample"]))
    print("Limitations:")
    for lim in entry["limitations"]:
        print(" -", lim)
    print()